<a href="https://colab.research.google.com/github/HowardChen0211/gisPy/blob/main/ProblemSets_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ProblemSet 0



### Research Question:

How do livability, affordability, and economic security vary across California County?


### Unit of Analysis
Counties in California (n = 58)

#### Install the package (If needed)

In [1]:
# !pip install geopandas
# !pip install mapclassify

#### Import the package from library

In [ ]:
import os, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd # # GIS / maps
import mapclassify  # required by geopandas for scheme="quantiles"

pd.set_option("display.float_format", "{:,.2f}".format)

import warnings
warnings.filterwarnings('ignore')

#### Load the Geographic Data
These datasets contain shapefile boundaries for CA State, counties and places from the US Census Bureau's 2023 MAF/TIGER database

In [ ]:
# The CA Boundaries data can be found at here https://catalog.data.gov/dataset/ca-geographic-boundaries

# -q = quiet; -O = output filename; saves file as CA_Counties_Boundaries.zip

# CA Counties Boundary dataset
! wget -q -O CA_Counties_Boundaries.zip https://data.ca.gov/dataset/e212e397-1277-4df3-8c22-40721b095f33/resource/b0007416-a325-4777-9295-368ea6b710e6/download/ca_counties.zip

# CA State Boundary dataset
! wget -q -O CA_State_Boundaries.zip https://data.ca.gov/dataset/e212e397-1277-4df3-8c22-40721b095f33/resource/3db1e426-fb51-44f5-82d5-a54d7c6e188b/download/ca_state.zip

# CA Places Boundary dataset
! wget -q -O CA_Places_Boundaries.zip https://data.ca.gov/dataset/e212e397-1277-4df3-8c22-40721b095f33/resource/436fc714-831c-4070-b44b-b06dcde6bf18/download/ca_places.zip


for z in ["CA_Counties_Boundaries.zip", "CA_State_Boundaries.zip", "CA_Places_Boundaries.zip"]:
  with zipfile.ZipFile(z, 'r') as zf:
    zf.extractall()

!ls # see what we got

In [ ]:
state = gpd.read_file("CA_State.shp")
counties = gpd.read_file("CA_Counties.shp")
cities = gpd.read_file("CA_Places.shp")

print("State:", state.shape, "| Counties:", counties.shape, "| Places:", cities.shape)

### State Boundary

In [ ]:
state.head() # see the first 5 rows

In [ ]:
state.info()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
state.plot(ax=ax, color="#e8e8e8", edgecolor="black", linewidth=0.6)
ax.set_title("California, State Boundary")
ax.set_axis_off()
plt.show()

As you can see, it shows the state of California

### Counties Boundary




In [ ]:
counties.head() # see the first 5 rows

In [ ]:
counties.info()

In [ ]:
# Convert the GEOID dtypes
counties["GEOID"] = counties["GEOID"].astype(int) # "06001" -> 6001, to match ACS FIPS

print("Dimensions:", counties.shape)
counties.head()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
counties.plot(ax=ax, color="#f5f5f5", edgecolor="#555555", linewidth=0.5)
ax.set_title("California, 58 counties")
ax.set_axis_off()
plt.show()

As you can see, it shows 58 counties boundary of California

### Cities Boundary

In [ ]:
cities.info()

The Places file mixes incorporated cities (CLASSFP starting with "C") and Census Designated Places (unincorporated, CLASSFP "U"). Only incorporated, active places are kept: Reduced from 1,521 to 482 incorporated cities

In [ ]:
mask = cities["CLASSFP"].str.startswith("C", na=False)
cities = cities.loc[mask, ["GEOID", "NAME", "CLASSFP", "geometry"]]

print("Incorporated cities:", len(cities))

In [ ]:
print("Dimensions:", cities.shape) # Reduced from 1,521 to 482 incorporated cities

# Convert the GEOID dtypes
cities["GEOID"] = cities["GEOID"].astype(int)

cities.head()

In [ ]:
cities["CLASSFP"].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
cities.plot(ax=ax, color="#2b6cb0", linewidth=0)
ax.set_title("California, 482 incorporated cities")
ax.set_axis_off()
plt.show()

As you can see, it shows the places of California excluding the Census Designated Places.
And there are three dense clusters(the Bay Area, the Los Angeles basin, and the San Diego coast)

# ProblemSet 1

### Map 1 - A map of counties and municipality boundaries in California

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))

state.plot(ax=ax, color="#EAF2F8", edgecolor="#2C3E50", linewidth=1.0)

counties.plot(ax=ax, color="none", edgecolor="#95A5A6", linewidth=0.45)

cities.plot(ax=ax, color="#E67E22", edgecolor="#e67e22", alpha=0.65, linewidth=0.35)

ax.set_title("California: 58 Counties and 482 Incorporated Cities", fontsize=14, fontweight="bold", color="#2C3E50")

ax.set_axis_off()

fig.text(0.10, 0.09, "Boundaries: U.S. Census Bureau TIGER/Line 2023 via California Open Data Portal", fontsize=7, color="#666666")

plt.show()

## ACS(American Community Survey) Data

Soure: https://www.socialexplorer.com/data-library/2 2020--2024 (5-Year-Estimates)

#### Load the data #1 (Population, Household Income, House Value)



In [ ]:
# population at County level
pop = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/Population_2024.csv", skiprows=[1])
pop.head()

In [ ]:
pop.info()

In [ ]:
# Median household income at County level
household_income = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/household_income_2024.csv", skiprows=[1])
household_income.rename(columns={"Median Household Income (In 2024 Inflation Adjusted Dollars)": "Median Household Income"}, inplace=True)
household_income.head()

In [ ]:
household_income.info()

In [ ]:
# Median owner-occupied house value at County level
house_val = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/housevalue_2024.csv", skiprows=[1])
house_val.head()

In [ ]:
house_val.info()

#### Clean the data

In [ ]:
census_house = (
    pop[["FIPS", "NAME", "Total Population", "Population Density (Per Sq. Mile)"]]
    .merge(household_income[["FIPS", "Median Household Income"]], on="FIPS", how="outer", validate="one_to_one")
    .merge(house_val[["FIPS", "Median House Value"]], on="FIPS", how="outer", validate="one_to_one")
)

num_cols = ["Total Population", "Population Density (Per Sq. Mile)", "Median Household Income", "Median House Value"]
census_house[num_cols] = census_house[num_cols].apply(pd.to_numeric, errors="coerce") # to numeric types, if missing values found will fill it NaN

In [ ]:
census_house.rename(columns={"Total Population": "TPOP",
                     "Population Density (Per Sq. Mile)": "POPDEN",
                     "Median Household Income": "MED_INC",
                     "Median House Value": "MED_HVAL"}, inplace = True)

In [ ]:
# Before
census_house.head()

In [ ]:
# change the NAME to make sure consistency
census_house["NAME"] = census_house["NAME"].str.split(",").str[0]

In [ ]:
# After
census_house.head()

## California Unemployment Data
Source: Social explorer
American Community Survey 2020--2024 (5-Year-Estimates) https://www.socialexplorer.com/reports/socialexplorer/en/report/bac15980-b308-11f1-9512-c38da70940e6

#### Load the data #2 (labor force and unemp)

In [ ]:
# Labor Force 16 Years and Over
unemployment = pd.read_csv("https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/CA_UnemploymentRate.csv")
unemployment.head()

In [ ]:
unemployment.info()

#### Clean the data

In [ ]:
# Choose the columns I need
unemployment = unemployment[["FIPS",
                             "NAME",
                             'Civilian Population in Labor Force 16 Years and Over',
                             'Civilian Population in Labor Force 16 Years and Over: Employed',
                             'Civilian Population in Labor Force 16 Years and Over: Unemployed']].iloc[1:, :]

# Rename the columns
unemployment.rename(columns={"NAME":"County",
                             "Civilian Population in Labor Force 16 Years and Over": "Labor Force",
                             "Civilian Population in Labor Force 16 Years and Over: Employed": "Employed",
                             'Civilian Population in Labor Force 16 Years and Over: Unemployed':"Unemployed"}, inplace=True)
# Convert the FIPS dtypes
unemployment["FIPS"] = unemployment["FIPS"].astype(int)

# get rid of ' County, California'
unemployment["County"] = unemployment["County"].str.replace(" County, California", "")

In [ ]:

unemployment.head()

In [ ]:
# Convert the data type
unemployment[['Labor Force', 'Employed', 'Unemployed']] = unemployment[['Labor Force', 'Employed', 'Unemployed']].astype(int)
unemployment['Rate'] = unemployment['Unemployed'] / unemployment['Labor Force'] * 100

unemployment.info()

In [ ]:
unemployment.head()

In [ ]:
print("rows:", len(unemployment), "| missing:", unemployment.isna().sum().sum(), "| duplicated", unemployment.duplicated().sum())

### Merge 1 and Merge 2 onto the county boundaries

Both joins use the 5-digit county FIPS code (`GEOID` = `FIPS`)

In [ ]:
def check_outer(left, right, left_on, right_on):
  check = pd.merge(left, right, left_on=left_on, right_on=right_on, how="outer", indicator=True)
  print(check["_merge"].value_counts().to_string(), "\n")

print("Merge 1:"); check_outer(counties, census_house, "GEOID", "FIPS")
print("Merge 2:"); check_outer(counties, unemployment, "GEOID", "FIPS")

In [ ]:
gdf = (
    counties
    .merge(census_house.drop(columns="NAME"), left_on="GEOID", right_on="FIPS", how="inner", validate="one_to_one")  # Merge 1
    .drop(columns="FIPS")
    .merge(unemployment, left_on="GEOID", right_on="FIPS", how="inner", validate="one_to_one")                       # Merge 2
    .drop(columns="FIPS")
)

gdf.shape

In [ ]:
gdf.info()

## Map 2 - where people, money, and expensive homes are

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7))

maps = [
    ('POPDEN', 'Population Density (Per Sq. Mile)'),
    ('MED_INC', 'Median Household Income'),
    ("MED_HVAL", "Median House Value")
]

for ax, (col, title) in zip(axes, maps):
    counties.plot(ax = ax, color='#2a2a2e', edgecolor='#999999', linewidth=0.3)
    gdf.plot(
        ax=ax,
        column=col, # Data to plot
        legend=True,
        cmap='YlOrRd', # color palette
        scheme='quantiles',
        k=5,
        linewidth=0.2,
        legend_kwds={'fmt': '{:,.0f}', 'loc': 'lower left',
                     'markerscale': 0.8, 'frameon': False, 'fontsize': 8}
    )
    ax.set_title(f'{title}\nCalifornia Counties (ACS 2020–2024)', fontsize=11)
    ax.set_axis_off()

fig.text(0.01, 0.02,
         'Source: U.S. Census Bureau, ACS 5-Year Estimates via Social Explorer;'
         '\nBoundaries: TIGER/Line 2024 Places',
         fontsize=7, color='#666666')

fig.tight_layout()
plt.show()

**Interpretation:** It shows that California's population density, household income, and housing prices are highly clustered in the Bay Area and the Los Angeles-San Diego along the coast. The San Francisco region ranks at the highest level in all three indicators, while the Central Valley and the northern inland counties are consistently lower.

### Housing Price-to-Income Ratio

In [ ]:
gdf["Price2Income"] = gdf["MED_HVAL"] / gdf["MED_INC"]  # price-to-income ratio (×)

gdf[["TPOP", "MED_INC", "MED_HVAL", "Price2Income", "Rate"]].describe().T

In [ ]:
# tiny labor forces
gdf.loc[gdf["Labor Force"] < 5000, ["NAME", "Labor Force", "Rate"]].sort_values("Labor Force")

**Interpretation:** Three counties (Alpine, Sierra, Modoc) have fewer than 5,000 people in the labor force. Sierra's 12.6% unemployment rests on about 157 unemployed people, so its true rate could be several points higher or lower.

I create the Functions that help for prettier maps, so that I can just call it.(See below)

In [ ]:
from matplotlib.patheffects import withStroke
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from shapely.geometry import box
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SRC_CES = "CalEnviroScreen 5.0, OEHHA (2026)"
SRC_ACS = "U.S. Census Bureau, ACS 2020–2024 5-Year Estimates via Social Explorer"
SRC_GEO = "Boundaries: U.S. Census Bureau TIGER/Line 2023 via California Open Data Portal"

def label_counties(ax, data, names=None, fontsize=5):
  '''write county names at the polygon centroids; names=None labels every county'''
  sub = data if names is None else data[data["NAME"].isin(names)]
  for pt, name in zip(sub.geometry.centroid, sub["NAME"]):
    ax.annotate(name, xy=(pt.x, pt.y), ha="center", va="center", fontsize=fontsize, color="black",
                path_effects=[withStroke(linewidth=2.5, foreground="white")])

def finish_map(ax, title, fontsize=13):
  '''state outline on top, title, no axes'''
  state.boundary.plot(ax=ax, color="#2C3E50", linewidth=0.8)
  ax.set_title(title, fontsize=fontsize, fontweight="bold", color="#2C3E50")
  ax.set_axis_off()

def source_note(fig, *sources, y=0.02):
  fig.text(0.02, y, "Source: " + "; ".join(sources) + "\n" + SRC_GEO, fontsize=7, color="#666666")

## Map 3 - Housing Burden: Price-to-Income Ratio

In [ ]:
# Source: https://geopandas.org/en/stable/docs/user_guide/mapping.html

fig, ax = plt.subplots(1, 1, figsize=(8, 10))

gdf.plot(
    ax=ax,
    column="Price2Income", # Data to plot
    cmap='YlOrRd', # color palette
    legend=True,
    scheme='quantiles', # k = 5, Number of classes defalut is 5
    linewidth=0.5,
    legend_kwds={"title": "Price2Income Ratio", 'fmt': '{:.1f}', 'loc': 'upper right',
                  'markerscale': 0.8, 'frameon': False, 'fontsize': 11, "title_fontsize": 13, 'edgecolor':"black"}
)

## labels
from matplotlib.patheffects import withStroke
# Get centroids of the polygon geometries to place the labels
centroids = gdf.geometry.centroid
for x, y, label in zip(centroids.x, centroids.y, gdf.NAME):
    ax.annotate(label, xy=(x, y), horizontalalignment='center',
    fontsize=5, color='black', #fontweight='bold',
    path_effects=[withStroke(linewidth=3,foreground='white')]) #ytext=(3, 3), textcoords="offset points",


ax.set_title("Price to Income Ratio",
             fontsize=20, pad=14)
fig.text(0.01, 0.02,
         'Source: U.S. Census Bureau, ACS 2020–2024 5-Year Estimates via Social Explorer;'
         '\nBoundaries: U.S. Census Bureau TIGER/Line 2023 via California Open Data Portal',
         fontsize=7, color='#666666')

ax.set_axis_off()
fig.tight_layout()
plt.show()

**Interpretation:** Housing affordability varies substantially across California counties, with higher price-to-income ratios indicating places where housing costs are large relative to local household incomes.

In [ ]:
gdf.nlargest(5, "Price2Income")[["NAME", "MED_INC", "MED_HVAL", "Price2Income"]]

In [ ]:
gdf.nsmallest(5, "Price2Income")[["NAME", "MED_INC", "MED_HVAL", "Price2Income"]]

A typical home in Marin costs about **10 years** of the typical household's income; in Modoc it costs under **4**.

## Map 4 - Job Burden: County Unemployment Rate

In [ ]:
# Source: https://geopandas.org/en/stable/docs/user_guide/mapping.html

fig, ax = plt.subplots(1, 1, figsize=(8, 10))

gdf.plot(
    ax=ax,
    column="Rate", # Data to plot
    cmap='YlOrRd', # color palette
    legend=True,
    scheme='quantiles', # k = 5, Number of classes defalut is 5
    linewidth=0.5,
    legend_kwds={"title": "Percent Unemployed", 'fmt': '{:.1f}%', 'loc': 'upper right',
                  'markerscale': 0.8, 'frameon': False, 'fontsize': 11, "title_fontsize": 13}
)

## labels
from matplotlib.patheffects import withStroke
# Get centroids of the polygon geometries to place the labels
centroids = gdf.geometry.centroid
for x, y, label in zip(centroids.x, centroids.y, gdf.NAME):
    ax.annotate(label, xy=(x, y), horizontalalignment='center',
    fontsize=5, color='black', #fontweight='bold',
    path_effects=[withStroke(linewidth=3, foreground='white')]) #ytext=(3, 3), textcoords="offset points"


ax.set_title("County Unemployment Rates",
             fontsize=20, pad=14)
fig.text(0.02, 0.015,
         "Source: Social Explorer; U.S. Census Bureau. Unemployment Rate for Civilian Population in Labor Force 16 Years and Over, 2024. Prepared by Social Explorer., "
         "\nBoundaries: U.S. Census Bureau TIGER/Line 2024",
         fontsize=7, color="#666666")

ax.set_axis_off()
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
plt.show()

**Interpretation:** Unemployment is not the same everywhere. It is highest in the Central Valley, Imperial County, and the North Coast (10-12.5%). It is lowest in the Bay Area and the eastern California.

# ProblemSet 2

## CalEnviroScreen 5.0 Data
Source: https://data.ca.gov/dataset/calenviroscreen-5-0 (OEHHA, released July 2026, census-tract level)

CalEnviroScreen 5.0 was released in 2026, but the data inside it is from the same period as my ACS data:
- `pm` is the average of 2022, 2023 and 2024.
- `ACS2024Pop` (tract population) comes from the ACS 2020–2024 5-year estimates, the same ACS release I used in PS1.

In [ ]:
# Load the data
ces_raw = pd.read_csv("https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/calenviroscreen50_070126.csv")

print(ces_raw.shape)
ces_raw.head()

In [ ]:
'''Data Dictionary Description
{
"tract": "Census Tract ID from 2020 Decennial Census",
"county": "California county that the census tract falls within",
"ACS2024Pop": "2024 5-year ACS population estimates in census tracts",
"pm": "Annual mean PM2.5 concentrations",
"ozone": "Mean of summer months (May–Oct.) of the daily maximum 8-hour ozone concentration",
"PollutionP": "Pollution Burden percentile",
"asthma": "Age-adjusted rate of emergency department visits for asthma"
}
'''

# keep it simple: read only the 7 columns needed
ces = ces_raw[["tract", "county", "ACS2024Pop", "pm", "ozone", "PollutionP", "asthma"]]

# treat any negative value as missing
num_cols = ["pm", "ozone", "PollutionP", "asthma"]
ces[num_cols] = ces[num_cols].mask(ces[num_cols] < 0)

print("Dimensions:", ces.shape)   # there's 9,106 tracts
ces.head()

In [ ]:
ces.info()

In [ ]:
# Missing values and zero-population tracts
# Based on the data dictionary: pm and ozone have no NAs, asthma has 33 NAs, 25 tracts have zero population
print(ces.isna().sum().to_string(), "\n")

print("Zero-population tracts:", (ces["ACS2024Pop"] == 0).sum())

**Interpretation:** A missing value usually means there was no monitor, no people, or the number was not reliable.

### Clean the data: census tracts to counties

CalEnviroScreen is at census-tract level, but my unit of analysis is the county, so the tracts have to be aggregated before the join.

In [ ]:
# county FIPS from the tract ID: 6001400100 -> 6001
ces["FIPS"] = ces["tract"] // 1_000_000

def pop_weighted_mean(df, col):
  # population-weighted county mean of `col`;
  # tracts with a missing value are skipped
  d = df.dropna(subset=[col])
  num = (d[col] * d["ACS2024Pop"]).groupby(d["FIPS"]).sum()
  pop_total = d.groupby("FIPS")["ACS2024Pop"].sum()
  return num / pop_total

ces_cty = pd.DataFrame({
    "PM25":   pop_weighted_mean(ces, "pm"),
    "OZONE":  pop_weighted_mean(ces, "ozone") * 1000,   # parts per billion, easier to read (0.045 ppm = 45 ppb)
    "ASTHMA": pop_weighted_mean(ces, "asthma"),
})

ces_cty = ces_cty.reset_index()

print("rows:", len(ces_cty), "| missing:", ces_cty.isna().sum().sum(), "| duplicated", ces_cty["FIPS"].duplicated().sum())
ces_cty.head()

In [ ]:
check_outer(counties, ces_cty, "GEOID", "FIPS") # Check the merge first!

gdf = (gdf.merge(ces_cty, left_on="GEOID", right_on="FIPS", how="left", validate="one_to_one").drop(columns="FIPS"))

gdf.info()

## Map 5 - Air Quality by County

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8.5))

panels = [
    ("PM25",  "PM2.5: fine particles\nannual mean (µg/m³)", "OrRd", "{:.1f}"),
    ("OZONE", "Ozone: summer mean of\ndaily max 8-hour (ppb)", "BuPu", "{:.1f}"),
]

for ax, (col, title, cmap, fmt) in zip(axes, panels):
  gdf.plot(
      ax=ax,
      column=col,
      cmap=cmap,
      scheme="quantiles", k=5,
      edgecolor="white", linewidth=0.4,
      legend=True,
      legend_kwds={"fmt": fmt, "loc": "upper right", "frameon": False, "fontsize": 9,
                   "title": "Quintiles", "title_fontsize": 10},
  )
  label_counties(ax, gdf, names=gdf.nlargest(10, col)["NAME"], fontsize=7)   # label only the 10 worst counties
  finish_map(ax, title)

fig.suptitle("Where is California's air the dirtiest?",
             fontsize=15, fontweight="bold", color="black", y=0.98)
source_note(fig, SRC_CES)
fig.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

**Interpretation:** The counties with the highest PM2.5 levels are mainly in the San Joaquin Valley and Los Angeles area. Mountains trap pollution from cars, farming, and factories, making the air worse. Coastal and northern counties have cleaner air. Ozone is higher inland because it forms when pollution reacts with sunlight.

## Map 6 - Housing burden, Unemployment, and PM2.5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7))

maps = [
    ('Price2Income', 'Housing Burden\nPrice-to-Income Ratio'),
    ('Rate', 'Job Burden\nUnemployment Rate (%)'),
    ('PM25', 'Air Quality\nPM2.5 (µg/m³)')
]

for ax, (col, title) in zip(axes, maps):
    gdf.plot(
        ax=ax,
        column=col, # Data to plot
        legend=True,
        cmap='YlOrRd', # same color palette for all three, darker = worse
        scheme='quantiles',
        k=5,
        edgecolor='white',
        linewidth=0.3,
        legend_kwds={'fmt': '{:.1f}', 'loc': 'lower left',
                     'markerscale': 0.8, 'frameon': False, 'fontsize': 8}
    )
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()

fig.suptitle("Three Burdens in California Counties (darker = worse)", fontsize=16, y=1.03)
fig.text(0.01, 0.02,
         'Source: U.S. Census Bureau, ACS 2020–2024 5-Year Estimates via Social Explorer; CalEnviroScreen 5.0 (OEHHA)'
         '\nBoundaries: U.S. Census Bureau TIGER/Line 2023 via California Open Data Portal',
         fontsize=7, color='#666666')

fig.tight_layout()
plt.show()

Housing burden is darkest along the coast, but unemployment and PM2.5 are both darkest in the San Joaquin Valley. So the valley has more affordable housing, but fewer jobs and dirtier air. Southern California (Los Angeles, Orange, San Diego) is dark on both housing burden and PM2.5, and the far north has high unemployment but clean air.

In [ ]:
### Do the three variables move together?

In [ ]:
# Correlation between the three variables (58 counties)
gdf[["Price2Income", "Rate", "PM25"]].corr()

In [ ]:
fig = make_subplots( rows=1, cols=3, subplot_titles=[
      "Price to Income vs Unemp Rate",
      "Price to Income vs PM2.5",
      "Unemp Rate vs PM2.5"])

# Price2Income vs Rate
fig.add_trace(
    go.Scatter(
        x=gdf["Rate"],
        y=gdf["Price2Income"],
        mode="markers",
        text=gdf.index,
        hovertemplate="Unemp Rate: %{x}<br>Price2Income: %{y}<extra></extra>"
    ),
    row=1, col=1
)

# Price2Income vs PM25
fig.add_trace(
    go.Scatter(
        x=gdf["PM25"],
        y=gdf["Price2Income"],
        mode="markers",
        text=gdf.index,
        hovertemplate="PM2.5: %{x}<br>Price2Income: %{y}<extra></extra>"
    ),row=1, col=2)

# Rate vs PM25
fig.add_trace(
    go.Scatter(
        x=gdf["PM25"],
        y=gdf["Rate"],
        mode="markers",
        text=gdf.index,
        hovertemplate="PM2.5: %{x}<br> Unemp Rate: %{y}<extra></extra>"
    ),row=1, col=3)

fig.update_layout(
    title="Relationships Between Housing Burden, Unemployment Rate, and PM2.5",
    height=450,
    width=1200,
    showlegend=False)

fig.show()

Housing burden and PM2.5 have almost no relationship (r = -0.04), so expensive counties are not cleaner or dirtier on average. Unemployment and PM2.5 have a moderate positive relationship (r = 0.38): counties with higher unemployment tend to have more PM2.5. With only 58 counties this shows a relationship, not a cause.

## Map 7 - Where do people actually breathe it? County centroids sized by population

In [ ]:
# Replace each county polygon with its center point
county_points = gdf.copy()
county_points["geometry"] = gdf.geometry.centroid

county_points[["NAME", "TPOP", "PM25", "geometry"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

# county boundaries in the background
counties.plot(ax=ax, color="#f2f2f2", edgecolor="#999999", linewidth=0.4)

# one circle per county: size = population, color = PM2.5
county_points.plot(
    ax=ax,
    column="PM25", # Data to plot
    cmap='YlOrRd', # color palette
    markersize=county_points["TPOP"] / 3000, # bigger circle = more people
    edgecolor="black",
    linewidth=0.5,
    alpha=0.85,
    legend=True,
    legend_kwds={"label": "PM2.5 (µg/m³)", "shrink": 0.5}
)

# label the 6 largest counties
big = county_points.nlargest(6, "TPOP")
for x, y, label in zip(big.geometry.x, big.geometry.y, big.NAME):
    ax.annotate(label, xy=(x, y), horizontalalignment='center',
    fontsize=8, color='black',
    path_effects=[withStroke(linewidth=3, foreground='white')])

ax.set_title("Population and PM2.5 by County\n(circle size = population)", fontsize=16)
fig.text(0.01, 0.02,
         'Source: U.S. Census Bureau, ACS 2020–2024 5-Year Estimates via Social Explorer; CalEnviroScreen 5.0 (OEHHA)'
         '\nBoundaries: U.S. Census Bureau TIGER/Line 2023 via California Open Data Portal',
         fontsize=7, color='#666666')

ax.set_axis_off()
fig.tight_layout()
plt.show()

In [ ]:
# How many people live in the 10 counties with the highest PM2.5?
top10 = gdf.nlargest(10, "PM25")

print("Population in the 10 highest PM2.5 counties:", top10["TPOP"].sum())
print("Population of California:", gdf["TPOP"].sum())
print("Share:", round(top10["TPOP"].sum() / gdf["TPOP"].sum() * 100, 1), "%")

The biggest circles are also dark. Los Angeles, Orange, San Bernardino and San Diego are all among the 10 counties with the highest PM2.5, and together those 10 counties have about 55% of California's population (21.8 of 39.3 million people). The large light-colored area in the north and east of Map 5 looks clean, but very few people live there.

# ProblemSet 3

## Let's dive into Los Angeles ~~

**Why Los Angeles?** I'm going to visit there 😃, and Los Angeles County has 9.8 million people, and in my county maps it was always dark. Expensive housing, dirty air, and unemployment above the state median. But in those maps the whole county is **one polygon with one number**. A county average can't tell whether the same neighborhoods carry all of these problems, or whether they fall on different people in different places.

**Research questions**
1. How does LA compare with the other 57 California counties?

2. Inside LA, which neighborhoods have the worst housing burden, economic security, and air quality?

3. Do these problems happen in the same neighborhoods? Who lives there?

## 1. Los Angeles vs. the other California counties

#### Add housing burden variable

In [ ]:
hb = ces_raw[["tract", "ACS2024Pop", "housingB"]].copy()
hb["housingB"] = hb["housingB"].mask(hb["housingB"] < 0)
hb["FIPS"] = hb["tract"] // 1_000_000                      # 6037101110 -> 6037

hb_cty = pop_weighted_mean(hb, "housingB").reset_index()
hb_cty.columns = ["FIPS", "HOUSINGB"]
hb_cty.head()

In [ ]:
check_outer(gdf, hb_cty, "GEOID", "FIPS") # Check the merge first! It's expected !!

gdf = gdf.merge(hb_cty, left_on="GEOID", right_on="FIPS", how="left").drop(columns="FIPS")

# Keep the columns I need
cols = ["NAME", "TPOP", "MED_INC", "MED_HVAL", "Price2Income", "HOUSINGB", "Rate", "PM25", "OZONE"]

cty = gdf[cols].copy()
cty.head()

### Where does LA rank?

In [ ]:
# rank 1 = the highest value among 58 counties
# https://www.geeksforgeeks.org/python/pandas-dataframe-rank/
ranks = cty.set_index("NAME").rank(ascending=False)

compare = pd.DataFrame({
    "Los Angeles": cty.set_index("NAME").loc["Los Angeles"],
    "CA median county": cty.median(numeric_only=True),
    "LA rank (1 = highest)": ranks.loc["Los Angeles"],
})
compare

**What surprised me:** LA does not have the most expensive homes in California. It ranks #11, while the Bay Area is much more expensive. However, when I compare home prices to income, LA ranks #4. This means LA's affordability problem is more about low income compared with home prices.

### Top 10 counties for each burden

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

bars = [
    ("Price2Income", "Price-to-Income Ratio"),
    ("HOUSINGB", "Housing Burden (% low-income households)"),
    ("Rate", "Unemployment Rate (%)"),
    ("PM25", "PM2.5 (µg/m³)")
]

for ax, (col, title) in zip(axes.flat, bars):
  top10 = cty.sort_values(col, ascending=False).head(10)
  colors = ["#E67E22" if name == "Los Angeles" else "#BDBDBD" for name in top10["NAME"]]
  # https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.barh.html
  ax.barh(top10["NAME"].str.replace(" County", ""), top10[col], color=colors)
  ax.invert_yaxis()   # highest on top
  ax.set_title(title, fontsize=12)

fig.suptitle("Top 10 California counties (orange = Los Angeles)", fontsize=15)
fig.text(0.01, -0.01, "Source: " + SRC_ACS + "; " + SRC_CES, fontsize=7, color="#666666")
fig.tight_layout()
plt.show()

LA is in the top 10 for three of the four burdens:
- **Price-to-income:** #4, the only Southern California county in the top 5.
- **Housing burden:** #1, clearly ahead of Santa Cruz.
- **PM2.5:** #5. The four above it (Tulare, Kings, Fresno, Kern) are all in the San Joaquin Valley.

But LA is not in the top 10 for unemployment. The highest unemployment is in rural and Central Valley counties (Sierra, Imperial, Lake, Kings).

## 2. Inside Los Angeles: census tract data

From here on, the unit of analysis is the **census tract**. I use the same CalEnviroScreen 5.0 file as in PS2 (`ces_raw`), but this time I keep every tract in LA County instead of averaging them into one county value.

In [ ]:
#### Load the data: LA tracts only

'''Data Dictionary Description from (CalEnviroScreen 5.0)
{
"approx_loc": "Approximate location (city or community name)",
"housingB":   "Percentage of households that are both low-income and severely burdened by housing costs",
"unemp":      "Percent of the population over 16 that is unemployed and eligible for the labor force",
"pov":        "Percent of the population living below two times the federal poverty level",
"pm":         "Annual mean PM2.5 concentrations",
"ozone":      "Mean of summer months (May-Oct.) of the daily maximum 8-hour ozone concentration",
"diesel":     "Diesel particulate matter emissions from on-road and non-road sources",
"asthma":     "Age-adjusted rate of emergency department visits for asthma (per 10,000)",
"edu, ling, pop_und10, pop_ov64, hisp, white, black, asian": "demographics, used in Section 5"
}
'''

keep = ["tract", "approx_loc", "ACS2024Pop",
        "housingB",                      # affordability
        "unemp", "pov",                  # economic security
        "pm", "ozone", "diesel",         # air quality
        "asthma",                        # health
        "edu", "ling", "pop_und10", "pop_ov64", "hisp", "white", "black", "asian"]

la = ces_raw.loc[ces_raw["county"] == "Los Angeles", keep].copy()

# treat any negative value as missing (same as PS2)
num_cols = keep[3:]
la[num_cols] = la[num_cols].mask(la[num_cols] < 0)

la["ozone"] = la["ozone"] * 1000   # ppm -> ppb, same as PS2

print("Dimensions:", la.shape)   # 2,495 tracts in LA County
la.head()

In [ ]:
la.info()

In [ ]:
print(la.isna().sum().to_string(), "\n")
print("Zero-population tracts:", (la["ACS2024Pop"] == 0).sum())

About 20-30 tracts are missing the housing and economic variables. Before dropping them, I want to see which tracts these are.

In [ ]:
# tracts with very few people
small = la[la["ACS2024Pop"] <= 500].sort_values("ACS2024Pop")
print("Tracts with fewer than 500 people:", len(small))
small[["tract", "approx_loc", "ACS2024Pop", "housingB", "unemp", "pov"]].head(10)

https://catalog.data.gov/dataset/2023-cartographic-boundary-file-shp-census-tract-for-united-states-1-500000

Almost all of these tract IDs start with **6037980...**. The code range 9800 through 9899 was enforced for those census tracts that contained little or no population and represented a relatively large special land use area such as a National Park, military installation, or a business/industrial park. Their percentages are not reliable. So I drop tracts with fewer than 500 people, and tracts with a missing value.

In [ ]:
la = la[la["ACS2024Pop"] >= 500].dropna()
print("Dimensions:", la.shape)

#### Load the Geographic Data: LA census tracts

In [ ]:
# Source: https://data.lacounty.gov/datasets/2020-census-tracts-4/about

! wget -q -O tracts_20.zip https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/2020_Census_Tracts.zip
zip_ref = zipfile.ZipFile('tracts_20.zip', 'r'); zip_ref.extractall(); zip_ref.close() #just unzipping
tracts = gpd.read_file('2020_Census_Tracts.shp')

tracts["GEOID"] = 6037000000 + tracts["CT20"].astype(int)   # "101110" -> 6037101110, to match CES
tracts = tracts.to_crs(epsg=2229)   # California State Plane Zone 5 (feet), better for LA than lat/lon

print("Dimensions:", tracts.shape)

In [ ]:
check_outer(tracts, la, "GEOID", "tract") # Check the merge first!

`left_only` are the tracts I just dropped (fewer than 500 people or missing data). `right_only` is 0, so I kept every CES tracts.

In [ ]:
la_gdf = tracts.merge(la, left_on="GEOID", right_on="tract", how="inner", validate="one_to_one").drop(columns="tract")

# centroid coordinates (lat/lon)
centroids = la_gdf.geometry.centroid.to_crs(epsg=4326)
la_gdf["lon"] = centroids.x
la_gdf["lat"] = centroids.y

# drop Santa Catalina Island (south of 33.6°N): far off the coast and would make the maps tiny
la_gdf = la_gdf[la_gdf["lat"] > 33.6]

print("Dimensions:", la_gdf.shape)
print("People:", la_gdf["ACS2024Pop"].sum())

In [ ]:
main_vars = ["housingB", "unemp", "pov", "pm", "ozone", "diesel"]

la_gdf[main_vars].describe().T

After cleaning there are **2,456 tracts** left. The numbers inside LA vary a lot more than the county averages.
- **Housing burden:** the typical tract is about 21%, but it goes from 0% to 69%.
- **Unemployment:** median 6.8%, but up to 36%.
- **Poverty:** from 0% to 94%, median 29%.
- **PM2.5:** from 5.4 to 13.2 µg/m³. The top of that range is higher than any county average in California (the worst county, Tulare, is 10.9).
- **Diesel:** the mean (0.14) is almost twice the median (0.08), so a few tracts have very high values.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

hists = [
    ("housingB", "Housing burden (%)"),
    ("unemp", "Unemployment (%)"),
    ("pov", "Poverty (%)"),
    ("pm", "PM2.5 (µg/m³)"),
    ("ozone", "Ozone (ppb)"),
    ("diesel", "log(Diesel PM)"),
]

for ax, (col, title) in zip(axes.flat, hists):
  values = la_gdf[col]
  if col == "diesel":
    values = np.log(values)   # diesel is extremely skewed, so I plot the log
  ax.hist(values, bins=40, color="#F5B041", edgecolor="white")
  ax.axvline(values.median(), color="black", linestyle="--", linewidth=1.5)
  ax.set_title(title)

fig.suptitle("Distribution of each variable across LA County census tracts", fontsize=14)
fig.text(0.01, -0.01, "Source: " + SRC_CES, fontsize=7, color="#666666")
fig.tight_layout()
plt.show()

Housing burden is my key affordability variable, so I start with it. It is roughly bell-shaped: most tracts are between 15% and 27%, with a right tail going up to 69% (skewness 0.46). There is no big pile-up at zero, so this variable separates neighborhoods well.

The others:
- **Unemployment** is right-skewed (1.24): most tracts are between 4.5% and 9.4%, with a long tail of a few tracts above 20%.
- **Poverty** is spread out almost evenly between 10% and 60%.
- **PM2.5** is **left**-skewed (-0.83): most of LA sits between 10 and 12 µg/m³, and the long tail is on the clean side. That tail is the north of the county (Antelope Valley, Santa Clarita), see the outliers below.
- **Ozone** has a big spike at 42–43 ppb and a second hump around 55–62. That looks like two different groups of tracts: the coast and the inland valleys. I'll check this on the map in Section 3.
- **Diesel** is extremely skewed (20.9). Even after taking the log, a few tracts are far to the right.

Because almost every variable is skewed or has more than one hump, I'll use **quantiles** (5 classes with the same number of tracts) for the maps in Section 3, like in PS1–PS2. With equal intervals, most tracts would end up in one or two colors.

### Who are the outliers?

In [ ]:
show = ["GEOID", "approx_loc", "ACS2024Pop", "lat", "lon"]

print("Highest housing burden")
print(la_gdf.nlargest(5, "housingB")[show + ["housingB", "pov"]].to_string(), "\n")

print("Highest unemployment")
print(la_gdf.nlargest(5, "unemp")[show + ["unemp", "pov"]].to_string(), "\n")

print("Highest diesel PM")
print(la_gdf.nlargest(3, "diesel")[show + ["diesel"]].to_string())

I checked them on Google Maps (lat/lon)

- **Housing burden:** the top three are all at 34.07°N, -118.45°W, which is **next to UCLA**. The 4th (52%) is at 34.02°N, -118.28°W, next to **USC**. Their poverty rates are 81-92%. These are most likely **students**: they have very low income, and paying high rents near campus. So the very highest housing burden values are partly a student effect, not only working families. I keep them, but this is a limitation of the variable.
- **Unemployment:** the top three (32-36%) are next to each other at 34.04°N, -118.24°W, which is **in Downtown LA**, the biggest concentration of homelessness in the city. Their poverty is 87-94%. This outlier is real.
- **Diesel:** the top two (10.9 and 6.8) are side by side at 34.22°N, -118.47°W, next to **Van Nuys Airport**, one of the busiest airports in the country. That is a real emission source too.

So the outliers are not data errors. They are some of the most unusual places in LA, and they tell me the extremes of each variable can mean different things.

## References

**Data**
1. U.S. Census Bureau, TIGER/Line 2023 — CA state, county, and places boundaries, via California Open Data Portal: https://data.ca.gov/dataset/ca-geographic-boundaries
2. Social Explorer; U.S. Census Bureau. Total Population (A00001), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A00001
3. Social Explorer; U.S. Census Bureau. Population Density (A00002), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A00002
4. Social Explorer; U.S. Census Bureau. Median Household Income (A14006), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A14006
5. Social Explorer; U.S. Census Bureau. Median House Value (A10036), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A10036
6. Social Explorer; U.S. Census Bureau. Employment Status for Civilian Population in Labor Force 16+ (A17005), ACS 2024 5-Year. https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A17005

7. List of counties in California:
https://en.wikipedia.org/wiki/List_of_counties_in_California

8. List of census-designated places in California:
https://en.wikipedia.org/wiki/List_of_census-designated_places_in_California

9. California Office of Environmental Health Hazard Assessment (OEHHA). CalEnviroScreen 5.0 (July 2026), via California Open Data Portal: https://data.ca.gov/dataset/calenviroscreen-5-0
    - Results CSV: https://data.ca.gov/dataset/72b28c84-ceac-4886-9f71-d422470d2223/resource/c4e277e0-cf23-4a8f-b07e-c8544c5d3d2b/download/calenviroscreen50_070126.csv
    - Shapefile: https://data.ca.gov/dataset/72b28c84-ceac-4886-9f71-d422470d2223/resource/e3d16016-1828-424f-85a6-f7731033d338/download/calenviroscreen50results_f_070126.shp.zip
    - Data dictionary: https://data.ca.gov/dataset/72b28c84-ceac-4886-9f71-d422470d2223/resource/31ddb21a-44bb-4cb0-81d8-6e2f80ee359d/download/final-calenviroscreen-5.0-data-dictionary.pdf
    
10. GeoPandas user guide, mapping: https://geopandas.org/en/stable/docs/user_guide/mapping.html

11. GeoPandas user guide, set operations with overlay: https://geopandas.org/en/stable/docs/user_guide/set_operations.html

12. Joshua Stevens, "Bivariate Choropleth Maps: A How-to Guide": https://www.joshuastevens.net/cartography/make-a-bivariate-choropleth-map/